In [7]:
import pandas as pd 
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
import torch

In [8]:
df = pd.read_csv('../../IEEE-CIS-Fraud-Detection/Data/04_Encoder/encoder.csv')

In [9]:
đf = df[:70000]

In [10]:
# X: tất cả các cột trừ cột TARGET
X = df.drop("binary_flags__TARGET", axis=1)

# y: cột TARGET
y = df["binary_flags__TARGET"]

In [11]:
# 1. Chia tập data thành bộ train và test với tỉ lệ 80/20 (Giữ nguyên phần chuẩn của bạn)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    random_state=42, 
    train_size=0.8, 
    shuffle=True,
    stratify=y 
)

print("--- Trước khi Oversampling ---")
print("Tập Train gốc:", Counter(y_train))
print("Tập Test gốc (GIỮ NGUYÊN, KHÔNG ĐỤNG VÀO):", Counter(y_test))
print("-" * 30)

# THỬ NGHIỆM 1: Thiết lập tỉ lệ thành 50/50
# Ý nghĩa: Số lượng mẫu lớp 1 sẽ được nhân bản lên bằng 100% số lượng lớp 0
ros_5050 = RandomOverSampler(sampling_strategy=1.0, random_state=42)
X_train_res1, y_train_res1 = ros_5050.fit_resample(X_train, y_train)

print("Sau oversampling 50/50 (trên tập Train):", Counter(y_train_res1))


# THỬ NGHIỆM 2: Thiết lập tỉ lệ thành 60/40
# Công thức tính: 40% / 60% = 0.6666... (khoảng 0.67)
# Ý nghĩa: Số lượng lớp 1 sau khi tăng sẽ bằng ~67% số lượng lớp 0
ros_6040 = RandomOverSampler(sampling_strategy=0.667, random_state=42)
X_train_res2, y_train_res2 = ros_6040.fit_resample(X_train, y_train)

print("Sau oversampling 60/40 (trên tập Train):", Counter(y_train_res2))


# THỬ NGHIỆM 3: Thiết lập tỉ lệ thành 70/30
# Công thức tính: 30% / 70% = 0.4285... (khoảng 0.429)
# Ý nghĩa: Số lượng lớp 1 sau khi tăng sẽ bằng ~42.9% số lượng lớp 0
ros_7030 = RandomOverSampler(sampling_strategy=0.429, random_state=42)
X_train_res3, y_train_res3 = ros_7030.fit_resample(X_train, y_train)

print("Sau oversampling 70/30 (trên tập Train):", Counter(y_train_res3))

--- Trước khi Oversampling ---
Tập Train gốc: Counter({0: 81263, 1: 19853})
Tập Test gốc (GIỮ NGUYÊN, KHÔNG ĐỤNG VÀO): Counter({0: 20316, 1: 4963})
------------------------------
Sau oversampling 50/50 (trên tập Train): Counter({1: 81263, 0: 81263})
Sau oversampling 60/40 (trên tập Train): Counter({0: 81263, 1: 54202})
Sau oversampling 70/30 (trên tập Train): Counter({0: 81263, 1: 34861})


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

***
### Tỉ lệ giữ nguyên không thay đổi 

In [13]:
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)
# Xuất bảng kết quả đánh giá các model
print(models)

Large dataset detected (101116 samples). Training all models may take a long time. Consider using a subset or setting max_models/timeout.


                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                          
BaggingClassifier            0.931801           0.834766  0.884683  0.926937   
AdaBoostClassifier           0.929348           0.820601  0.914268  0.923148   
DecisionTreeClassifier       0.880691           0.820555  0.820555  0.881796   
RandomForestClassifier       0.926382           0.817461  0.907083  0.920111   
NearestCentroid              0.672139           0.681745  0.746599  0.704587   
BernoulliNB                  0.753827           0.678359  0.764913  0.766999   
ExtraTreesClassifier         0.866767           0.675231  0.862328  0.842665   
Perceptron                   0.744333           0.635071  0.710926  0.753207   
PassiveAggressiveClassifier  0.750504           0.630992  0.710820  0.756465   
ExtraTreeClassifier          0.749713           0.612685  0.612685  0.751951   
SVC                          0.832865   

***
### Tỉ lệ 50 / 50

***
### Tỉ lệ  60 / 40

In [ ]:
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train_res2, X_test, y_train_res2, y_test)
# Xuất bảng kết quả đánh giá các model
print(models)

***
### Tỉ lệ 70 / 30

In [ ]:
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train_res3, X_test, y_train_res3, y_test)
# Xuất bảng kết quả đánh giá các model
print(models)